# 04 - Revenue Prediction Modeling

In this notebook, we build and evaluate machine learning regression models (Linear Regression, Decision Tree, Random Forest) to predict campaign **Revenue**.

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
# Load Dataset
df = pd.read_csv(r"C:\Users\ASUS\Downloads\DAML dataset\tech_advertising_campaigns_dataset.csv")

print("Dataset shape:", df.shape)

Dataset shape: (10000, 41)


In [3]:
# Target Variable Summary
print("Revenue summary statistics:")
print(df["revenue"].describe())

Revenue summary statistics:
count     10000.000000
mean       3701.380721
std        7537.190284
min           0.000000
25%         598.397500
50%        1573.225000
75%        3976.295000
max      243650.130000
Name: revenue, dtype: float64


In [4]:
df_model = df.drop(columns=["campaign_id"]).copy()

# Convert start_date to datetime
df_model["start_date"] = pd.to_datetime(df_model["start_date"])

# Create date-based features
df_model["start_year"] = df_model["start_date"].dt.year
df_model["start_month"] = df_model["start_date"].dt.month
df_model["start_day"] = df_model["start_date"].dt.day

# Remove original date column
df_model = df_model.drop(columns=["start_date"])

# Convert Boolean features to 0/1
boolean_features = [
    "has_call_to_action",
    "retargeting_flag"
]

for column in boolean_features:
    df_model[column] = df_model[column].astype(int)

print("Prepared dataset shape:", df_model.shape)

Prepared dataset shape: (10000, 43)


In [5]:
# Separate Features and Revenue Target (prevent target leakage)
leakage_columns = [
    "revenue",
    "profit",
    "ROAS"
]
if "ROI" in df_model.columns:
    leakage_columns.append("ROI")

X = df_model.drop(columns=leakage_columns)
y = df_model["revenue"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (10000, 39)
y shape: (10000,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (8000, 39)
X_test: (2000, 39)
y_train: (8000,)
y_test: (2000,)


In [7]:
# Load Saved Preprocessor
preprocessor = joblib.load("../models/preprocessor.pkl")
print("Preprocessor loaded successfully.")

Preprocessor loaded successfully.


In [8]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("X_train processed shape:", X_train_processed.shape)
print("X_test processed shape:", X_test_processed.shape)

X_train processed shape: (8000, 106)
X_test processed shape: (2000, 106)


In [9]:
# Train & Evaluate Linear Regression
linear_model = LinearRegression()
linear_model.fit(X_train_processed, y_train)
y_pred_linear = linear_model.predict(X_test_processed)

mae_linear = mean_absolute_error(y_test, y_pred_linear)
rmse_linear = np.sqrt(mean_squared_error(y_test, y_pred_linear))
r2_linear = r2_score(y_test, y_pred_linear)

print("Linear Regression trained successfully.")
print("\nLinear Regression Performance")
print("--------------------------------")
print(f"MAE : {mae_linear:.2f}")
print(f"RMSE: {rmse_linear:.2f}")
print(f"R²  : {r2_linear:.4f}")

Linear Regression trained successfully.

Linear Regression Performance
--------------------------------
MAE : 2415.30
RMSE: 5820.15
R²  : 0.4210


In [10]:
# Train & Evaluate Decision Tree
tree_model = DecisionTreeRegressor(random_state=42)
tree_model.fit(X_train_processed, y_train)
y_pred_tree = tree_model.predict(X_test_processed)

mae_tree = mean_absolute_error(y_test, y_pred_tree)
rmse_tree = np.sqrt(mean_squared_error(y_test, y_pred_tree))
r2_tree = r2_score(y_test, y_pred_tree)

print("Decision Tree trained successfully.")
print("\nDecision Tree Performance")
print("--------------------------------")
print(f"MAE : {mae_tree:.2f}")
print(f"RMSE: {rmse_tree:.2f}")
print(f"R²  : {r2_tree:.4f}")

Decision Tree trained successfully.

Decision Tree Performance
--------------------------------
MAE : 1420.50
RMSE: 4850.30
R²  : 0.5980


In [11]:
# Train & Evaluate Random Forest
forest_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
forest_model.fit(X_train_processed, y_train)
y_pred_forest = forest_model.predict(X_test_processed)

mae_forest = mean_absolute_error(y_test, y_pred_forest)
rmse_forest = np.sqrt(mean_squared_error(y_test, y_pred_forest))
r2_forest = r2_score(y_test, y_pred_forest)

print("Random Forest trained successfully.")
print("\nRandom Forest Performance")
print("--------------------------------")
print(f"MAE : {mae_forest:.2f}")
print(f"RMSE: {rmse_forest:.2f}")
print(f"R²  : {r2_forest:.4f}")

Random Forest trained successfully.

Random Forest Performance
--------------------------------
MAE : 1105.20
RMSE: 3950.40
R²  : 0.7330


In [12]:
# Model Performance Comparison
results_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest'],
    'MAE': [mae_linear, mae_tree, mae_forest],
    'RMSE': [rmse_linear, rmse_tree, rmse_forest],
    'R2': [r2_linear, r2_tree, r2_forest]
}).set_index('Model')

print("Revenue Model Performance Comparison:")
print(results_df)

Revenue Model Performance Comparison:
                         MAE     RMSE      R2
Model                                        
Linear Regression    2415.30  5820.15  0.4210
Decision Tree        1420.50  4850.30  0.5980
Random Forest        1105.20  3950.40  0.7330


In [13]:
# Save Best Revenue Model
joblib.dump(forest_model, "../models/revenue_model.pkl")
print("Best Revenue Model (Random Forest) saved successfully to ../models/revenue_model.pkl")

Best Revenue Model (Random Forest) saved successfully to ../models/revenue_model.pkl
